# Day 1 · Module 1: Intent & Assumption Variance

**Objective:** turn an underspecified ticket into an intent doc that pins the decisions a model would otherwise make silently, then measure how much independent planning runs diverge from the gaps you left open — both when you re-run the *same* model and when you swap the model (Haiku, Sonnet, Opus) underneath the exact same intent doc.


## Relevant Claude Code Docs
Review these before starting the module:
- [Best practices for Claude Code](https://code.claude.com/docs/en/best-practices)
- [How Claude remembers your project](https://code.claude.com/docs/en/memory)
- [Keep Claude working toward a goal](https://code.claude.com/docs/en/goal)

## 1 · The idea

This module runs an artifact-driven pipeline: **scenario → intent → plan**.

An intent doc exists to pin the decisions a model would otherwise assume for you, silently and inconsistently. If you skip it and go straight from a vague ticket to a plan, the model fills every gap on its own — and it may fill the same gap differently the next time you ask.

The way to see this happening is to run the planning step more than once from the same intent doc. If the intent really pinned the ambiguous points, the plans should look similar where it matters. Wherever the intent stayed silent, the plans are free to diverge — and that divergence is exactly the "assumption variance" this module is named for.


### Two ways a plan can vary

There are two independent reasons two plans built from the same intent can disagree. Keeping them apart is the whole skill:

| Axis | What you change between runs | What the divergence tells you |
|------|------------------------------|-------------------------------|
| **Sampling** | Nothing — same model, fresh session | Noise. The model rolled the dice differently on a gap you left open. |
| **Model** | Swap Haiku ↔ Sonnet ↔ Opus | Whether your intent is pinned *tightly enough that the model choice stops mattering*. |

A gap that only wobbles on the sampling axis is mild — one run happened to guess differently. A gap that moves the moment you change models is louder: different models carry different defaults, so an unpinned decision doesn't just vary randomly, it varies *systematically* with whatever model you happened to run. A well-pinned intent doc collapses both: even Haiku's plan agrees with Opus's on the points you nailed down. Whatever still disagrees across models is a map of what you left to the model's priors.


### Why the model axis matters

"Just use a good model" is not a substitute for an intent doc — but the model you pick is itself a silent assumption. Faced with `TRANSFER_LIMIT = 100` and no unit, a smaller/faster model tends to commit to a plausible default and move on; a larger model is more likely to surface the ambiguity, hedge, or pin more of the neighbouring gaps (currency, window, per-partner scope) without being asked. Neither behavior is *wrong* — but it means the same vague ticket produces measurably different plans depending on which model read it, and nobody wrote that decision down.

```mermaid
flowchart LR
    T["Vague ticket<br/>TRANSFER_LIMIT = 100"] --> I["intent.md<br/>(what you pin)"]
    I --> H["Haiku plan"]
    I --> S["Sonnet plan"]
    I --> O["Opus plan"]
    H --> D{"diverge?"}
    S --> D
    O --> D
    D -->|"on pinned points"| G["intent gap — fix the doc"]
    D -->|"nowhere"| W["intent held across models"]
```

The exercise runs the plan under all three models from the *identical* intent doc. Where the three plans agree, your intent did the work. Where they split, you're watching each model's defaults leak in — and that split is the thing to drive out of your intent doc, not out of your model choice.


### Grounding

Look at the real trap sitting in `src/contoso/config.py`:

```python
TRANSFER_LIMIT = 100
THROTTLE_PER_SECOND = 20
DEFAULT_WINDOW_SECONDS = 60
```

`TRANSFER_LIMIT = 100` has no unit attached. Is that a **$100 ceiling** per transfer, a **count of 100 transfers**, or **100 per some window**? The neighbouring constants don't resolve this — if anything they make it worse, because `THROTTLE_PER_SECOND = 20` and `DEFAULT_WINDOW_SECONDS = 60` imply two *different* candidate units for the same "100," and neither of those units is even the same *kind* of thing as a dollar cap.

In a payments system this is not a rounding error — it's the difference between a $100 ceiling and 100 transfers/day, which is a fraud-and-compliance decision, not a formatting detail. A plan that doesn't explicitly pin the unit of `TRANSFER_LIMIT` has already guessed on your behalf, and you won't find out which guess it made until the behavior in production surprises someone — a partner, an auditor, or both.

Run the cell below to make the ambiguity concrete: three numbers, no units, one open question.


### More gaps than the one you can see

The unit of `TRANSFER_LIMIT` is the gap you can *see* just by reading `config.py`. A real payments limit carries several more gaps that don't show up until someone asks the question out loud. These are **candidates to consider, not answers to fill in** — you decide which of them actually apply to this ticket:

- **Currency.** Is the limit in USD only, or does it need to hold across   multiple settlement currencies?
- **Cross-border movement.** Does an international transfer count the same   as a domestic one, or does it draw down the limit differently (or not at   all)?
- **Holds vs. posted transfers.** Do authorization holds count toward the   limit the moment they're placed, or only once a transfer actually posts   to the ledger?
- **Per-account or per-partner.** Does the limit apply to a single account,   or to everything a partner API key touches across all of its accounts?
- **Rolling window or calendar day.** Is "100 per day" a rolling 24-hour   window, or does it reset at midnight regardless of when the count started?
- **Weekends and settlement days.** Does a transfer placed Friday night and   settled Monday count against Friday's window, Monday's, or both?

The scenario ticket only names a few of these explicitly. Part of the exercise is noticing which additional candidates from this list are worth pinning for *this* ticket, and which are genuinely out of scope for v1.


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` later for both the exercise and the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


With `proj` set, confirm the constants directly:


In [ ]:
from contoso import config

print("TRANSFER_LIMIT        =", config.TRANSFER_LIMIT)
print("THROTTLE_PER_SECOND    =", config.THROTTLE_PER_SECOND)
print("DEFAULT_WINDOW_SECONDS =", config.DEFAULT_WINDOW_SECONDS)


## 2 · Your exercise

Work through these steps in order:

1. Read the scenario ticket at `scenarios/m1-transfer-limit.md`. It's the underspecified request as it arrived from the business — resist the urge to "fix" it in your head before you've written anything down.
2. **Capture a baseline, before you write anything else.** In a fresh Claude Code session, give it *only* the scenario ticket — no intent doc, no template — and ask it to write a technical plan for implementing the request. Save the output to `artifacts/m1/plan.baseline.md`. This is what a model does when nobody has pinned the ambiguities yet; every later plan gets compared against it.
3. Hand-write `artifacts/m1/intent.md`, starting from `templates/intent.md`. Pin **≥4 gaps**, including a decision on the `TRANSFER_LIMIT` unit. Your intent doc must include an explicit "what this feature explicitly does NOT do" section — the boundary you draw is as important as the scope you claim.
4. In Claude Code, ask it to turn your intent doc into a technical plan **as a written document** — this step is about producing a planning *artifact*, not about invoking Claude Code's plan-mode feature. Plan mode is for Claude to plan *its own next action* before executing; it isn't a mechanism for generating a persisted plan file, so it's the wrong tool here. A prompt along these lines works:

   > Read `artifacts/m1/intent.md` and `templates/plan.md`. Write a technical plan for this feature to `artifacts/m1/plan.<model>.md`, following the template's structure. Do not write or modify any implementation code — this is a planning artifact only.

   Do this **once per model**, each in a fresh session so nothing leaks between runs, feeding the *same* `intent.md` every time. Switch models with `/model` before prompting, and point each output at the matching file:
   - `/model opus` → `artifacts/m1/plan.opus.md`
   - `/model sonnet` → `artifacts/m1/plan.sonnet.md`
   - `/model haiku` → `artifacts/m1/plan.haiku.md`

5. **Optional (separates the two axes):** run Opus a second time in a fresh session → `artifacts/m1/plan.opus.2.md`. Comparing the two Opus runs to each other isolates *sampling* noise; comparing Opus to Haiku/Sonnet isolates the *model* axis. The self-check picks up whichever `plan.*.md` files you produce (and still accepts the older `plan.md` / `plan.2.md` pair), plus `plan.baseline.md` if present.


### Vibes vs. mechanical: a worked example

"Acceptance criteria must be executable" is easy to say and easy to write past. Here's the same requirement written both ways, side by side:

- **Vibes AC:** "Transfer limits should be fair and shouldn't get in   legitimate partners' way."
  No test can pass or fail this. "Fair" isn't observable state — it's a   feeling about the system, and two reviewers could disagree about whether   any given behavior satisfies it.
- **Mechanical AC:** "A 101st transfer attempted by the same partner within   a rolling 24-hour window returns HTTP 429."
  This is checkable by a test today, without asking anyone what they meant:   post 100 transfers, post a 101st, assert the status code.

Notice the mechanical version had to *resolve* several of the gaps above — count vs. dollars, per-partner scope, rolling window, 429 as the on-limit behavior — before it could be written at all. That's not a coincidence: an acceptance criterion can only be mechanically checkable once its underlying assumptions are pinned. Vague ACs and unpinned gaps are the same problem wearing two different hats.


### What good looks like

Compare your intent doc against `reference/m1/intent.EXAMPLE.md`. That file is a **bar to compare against, not an answer to copy** — several of its hardest decisions are deliberately left as `[you decide]` placeholders. A strong intent doc is concrete exactly where the reference is blank.

Acceptance criteria must be **executable** — a test or a careful reviewer could check each one mechanically and fail it — not vibes like "the limit should be fair." (See the worked contrast above.)

The plans from step 3 should diverge specifically on the points your intent left open. That divergence isn't a bug in the exercise — it *is* the lesson: every place the plans disagree is a gap your intent doc failed to pin. Pay special attention to gaps that move when you *change models* rather than just re-run one: those aren't sampling noise, they're each model's default leaking into a decision you never wrote down.

Common failure modes:
- An intent doc with no "does NOT" section — scope creeps invisibly.
- Acceptance criteria that describe aspirations ("robust," "fair") instead   of observable, checkable outcomes.
- All the plans come out identical — not because you pinned everything well,   but because the intent (or the models) silently pre-decided every open   question the same way, so nothing was left to vary.
- Blaming the divergence on "Haiku being weaker" and reaching for a bigger   model instead of fixing the intent doc. A stronger model hides the gap;   it doesn't close it.

This intent doc is also the input the next module builds on: M2's reusable command takes an intent doc like this one and turns the scenario → intent → plan pipeline into something you invoke instead of re-explaining by hand.


### Verify your work

This checklist is advisory, not a gate — it just reflects back what it finds in `artifacts/m1/`. It's safe to run before you've written anything (the intent/plan checks below just report what's missing).

Once you have plans to compare, the gap analysis prefers to ask Claude Code itself to read each full plan and judge whether it addresses each candidate gap — not a keyword scan. That needs the `claude` CLI authenticated inside the devcontainer (or an `ANTHROPIC_API_KEY` set for it), and makes a small number of API calls the first time it sees a given plan file (cached by content hash afterward, so re-running is free unless a plan changed). If `claude` isn't available or isn't authenticated, both cells fall back to a deterministic keyword scan automatically and say so — lower-confidence (paraphrases it doesn't recognize read as silent), but never a silent failure.


In [ ]:
import pathlib, subprocess, json, hashlib, os

UNIT_TOKENS = [
    "dollar", "$", "count", "per day", "per transfer",
    "per second", "per-second", "per minute", "per-minute",
]

def _load_dotenv(path):
    """Load KEY=VALUE lines from a .env file into os.environ (without
    overwriting anything already set), so subprocess calls to `claude`
    inherit ANTHROPIC_API_KEY even though Jupyter itself didn't load it."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

m1 = pathlib.Path(proj) / "artifacts" / "m1"
_load_dotenv(pathlib.Path(proj) / ".env")
intent = m1 / "intent.md"
if intent.exists():
    t = intent.read_text().lower()
    print(f"[x] intent.md present ({len(t.split())} words)")
    print(f"  [{'x' if 'does not' in t or 'not do' in t else ' '}] has an explicit 'does NOT' scope section?")
    print(f"  [{'x' if any(tok in t for tok in UNIT_TOKENS) else ' '}] pinned the TRANSFER_LIMIT unit? (any of: {', '.join(UNIT_TOKENS)})")
    print(f"  [{'x' if 'acceptance' in t or 'ac1' in t or 'criteria' in t else ' '}] states acceptance criteria?")
else:
    print("[ ] artifacts/m1/intent.md missing — write it from templates/intent.md")

# --- discover plans ---------------------------------------------------------
# Prefer the model-named files (plan.opus.md, plan.sonnet.md, plan.haiku.md,
# plus any extra runs like plan.opus.2.md and the no-intent plan.baseline.md);
# fall back to the legacy pair.
PREFERRED_ORDER = ["baseline", "opus", "sonnet", "haiku"]

def label_for(path):
    """baseline / opus / sonnet / haiku / opus.2 / <stem> from a plan.*.md filename."""
    if path.name == "plan.md":
        return "run1"
    if path.name == "plan.2.md":
        return "run2"
    return path.name[len("plan."):-len(".md")] or "plan"

def sort_key(path):
    lab = label_for(path)
    base = lab.split(".")[0]
    rank = PREFERRED_ORDER.index(base) if base in PREFERRED_ORDER else len(PREFERRED_ORDER)
    return (rank, lab)

model_plans = sorted(
    (p for p in m1.glob("plan.*.md") if p.name != "plan.2.md"),
    key=sort_key,
)
if model_plans:
    plans = model_plans
    if (m1 / "plan.2.md").exists():
        plans.append(m1 / "plan.2.md")
else:
    # legacy layout: same-model twice
    plans = [p for p in (m1 / "plan.md", m1 / "plan.2.md") if p.exists()]

if len(plans) < 2:
    print("\n[ ] need at least two plans for the variance check.")
    print("    Produce artifacts/m1/plan.opus.md, plan.sonnet.md, plan.haiku.md")
    print("    (plus optionally plan.baseline.md — or the legacy plan.md + plan.2.md).")
else:
    labels = [label_for(p) for p in plans]
    print(f"\nComparing {len(plans)} plans: {', '.join(labels)}")
    if "baseline" in labels:
        print("  ('baseline' was written from the scenario ticket alone, with no intent doc — "
              "expect it to diverge from the others on most gaps. That contrast is the point: "
              "see how much LESS the intent-driven plans disagree with each other.)")

    # --- gap definitions ----------------------------------------------------
    # Each gap carries both an LLM judging question AND a keyword fallback, so
    # the exact same GAP_SPECS drives whichever judging method is available.
    JUDGE_MODEL = "opus"  # fixed regardless of which model wrote the plan, so
                          # the judge is never grading its own (or a sibling's)
                          # homework
    CACHE_PATH = m1 / ".judge_cache.json"

    GAP_SPECS = [
        ("unit", "What unit is TRANSFER_LIMIT measured in?",
         [("dollars", ["$", "dollar", "cent"]),
          ("count", ["count", "number of", "per transfer", "transfers per"])]),
        ("currency", "Does the limit apply to a single currency or multiple?",
         [("USD-only", ["usd", "us dollar", "dollar only", "single currency"]),
          ("multi-currency", ["multi-currenc", "multiple currenc", "eur", "settlement currenc"])]),
        ("cross-border", "Does an international transfer count differently than a domestic one?",
         [("domestic-only", ["domestic only", "no cross-border", "not international"]),
          ("counts-the-same", ["cross-border", "cross border", "international"]),
          ("differs-by-country", ["by country", "by-country", "region"])]),
        ("holds/posted", "Do authorization holds count toward the limit, or only posted transfers?",
         [("on-hold", ["when placed", "at authorization", "on placement", "hold"]),
          ("on-posted", ["posted", "posting", "once posted", "ledger"])]),
        ("scope", "Is the limit per-account, or per-partner (covering all of a partner's accounts)?",
         [("per-partner", ["per-partner", "per partner", "partner api key", "partner key", "api key"]),
          ("per-account", ["per-account", "per account", "single account"])]),
        ("window", "Is the limit a rolling window, or does it reset on a calendar boundary?",
         [("rolling", ["rolling", "24-hour", "24 hour", "last 24"]),
          ("calendar", ["calendar", "midnight", "resets"])]),
        ("weekend", "Does the plan address weekend/settlement-day timing edge cases?",
         [("addressed", ["weekend", "settlement day", "business day", "friday", "monday"])]),
    ]

    def _load_cache():
        if CACHE_PATH.exists():
            try:
                return json.loads(CACHE_PATH.read_text())
            except Exception:
                return {}
        return {}

    def _save_cache(cache):
        CACHE_PATH.write_text(json.dumps(cache, indent=2))

    def _run_claude_judge(plan_text, model):
        gap_lines = "\n".join(
            f'{i+1}. "{name}" — {question} Valid answers: '
            f'{", ".join(opt for opt, _ in options)}, or other.'
            for i, (name, question, options) in enumerate(GAP_SPECS)
        )
        prompt = (
            "You are grading a technical plan for a payments feature (transfer limits) on "
            "whether it explicitly makes each of the following decisions. Judge ONLY from "
            "what the plan text actually says — do not assume or infer beyond it.\n\n"
            f"{gap_lines}\n\n"
            "Respond with ONLY a JSON object (no markdown fences, no other text). For each "
            "gap name above, include a key with this exact shape:\n"
            '{"addressed": true|false, "decision": "<one of the valid answers, or a short '
            "phrase if none fit>\", \"evidence\": \"<short quote or 1-sentence paraphrase from "
            "the plan, or '' if not addressed>\"}\n\n"
            "Plan text:\n<<<\n" + plan_text + "\n>>>"
        )
        proc = subprocess.run(
            ["claude", "-p", prompt, "--model", model, "--output-format", "json"],
            capture_output=True, text=True, timeout=120,
        )
        if proc.returncode != 0:
            raise RuntimeError(f"claude exited {proc.returncode}: {proc.stderr.strip()[:300]}")
        try:
            envelope = json.loads(proc.stdout)
            raw = envelope["result"] if isinstance(envelope, dict) and "result" in envelope else envelope
        except json.JSONDecodeError:
            raw = proc.stdout
        text = raw if isinstance(raw, str) else json.dumps(raw)
        text = text.strip()
        if text.startswith("```"):
            text = text.split("\n", 1)[1] if "\n" in text else text
            if text.endswith("```"):
                text = text[:-3]
            text = text.strip()
        return json.loads(text)

    def _keyword_judge(plan_text):
        """Deterministic fallback: first keyword match per gap, else silent.
        Lower confidence than the LLM judge — paraphrases the keyword list
        doesn't cover will read as silent even if the plan addressed the gap."""
        low = plan_text.lower()
        out = {}
        for name, _, options in GAP_SPECS:
            decision = ""
            for opt, kws in options:
                if any(k in low for k in kws):
                    decision = opt
                    break
            out[name] = {"addressed": bool(decision), "decision": decision, "evidence": ""}
        return out

    def judge_plan(path):
        text = path.read_text()
        key = f"{path.name}:{hashlib.sha256(text.encode()).hexdigest()[:16]}"
        cache = _load_cache()
        if key in cache:
            return cache[key]
        try:
            result = _run_claude_judge(text, JUDGE_MODEL)
            result["_method"] = "llm"
        except FileNotFoundError:
            result = _keyword_judge(text)
            result["_method"] = "keyword"
            result["_fallback_reason"] = "`claude` CLI not found on PATH"
        except Exception as e:
            result = _keyword_judge(text)
            result["_method"] = "keyword"
            result["_fallback_reason"] = str(e)[:200]
        cache[key] = result
        _save_cache(cache)
        return result

    results, errors = {}, {}
    for lab, p in zip(labels, plans):
        try:
            results[lab] = judge_plan(p)
        except Exception as e:
            errors[lab] = str(e)

    if errors:
        print("\n  Could not judge (even the keyword fallback failed to read the file): "
              + "; ".join(f"{l} — {e}" for l, e in errors.items()))

    fallback_labels = [l for l in results if results[l].get("_method") == "keyword"]
    if fallback_labels:
        print(f"\n  Note: {', '.join(fallback_labels)} judged by keyword scan, not the LLM "
              "judge (claude CLI unavailable/unauthenticated — "
              + "; ".join(f"{l}: {results[l].get('_fallback_reason', '?')}" for l in fallback_labels)
              + "). Keyword results are lower-confidence: a paraphrase the list doesn't "
              "recognize reads as silent even if the plan actually addressed it.")

    usable = [l for l in labels if l in results]
    if len(usable) < 2:
        print("\n  Fewer than two plans could be judged — stopping here rather than "
              "comparing partial/empty data.")
    else:
        wln = max(len("gap"), max(len(name) for name, _, _ in GAP_SPECS))
        header = "gap".ljust(wln) + " | " + " | ".join(l.center(10) for l in usable)
        print(f"\nGap addressed?  (x = addressed, . = silent; LLM-judged by {JUDGE_MODEL} "
              "unless noted above)\n")
        print(header)
        print("-" * len(header))
        evidence_notes = []
        for name, _, _ in GAP_SPECS:
            cells_out, marks, decisions = [], [], []
            for l in usable:
                g = results[l].get(name, {})
                addressed = bool(g.get("addressed"))
                marks.append(addressed)
                decisions.append(g.get("decision") if addressed else None)
                cells_out.append(("x" if addressed else ".").center(10))
            row = name.ljust(wln) + " | " + " | ".join(cells_out)
            split = len(set(marks)) > 1 or len(set(d for d in decisions if d)) > 1
            if split:
                row += "   <- split"
                for l in usable:
                    g = results[l].get(name, {})
                    if g.get("evidence"):
                        evidence_notes.append(f"    {name} / {l}: {g.get('decision')!r} — \"{g['evidence']}\"")
            print(row)
        addressed_counts = {l: sum(1 for name, _, _ in GAP_SPECS if results[l].get(name, {}).get("addressed")) for l in usable}
        print("\ngaps addressed: " + ", ".join(f"{l} {addressed_counts[l]}/{len(GAP_SPECS)}" for l in usable))
        print("  Rows marked '<- split' are gaps where the plans disagree (or one is silent "
              "and another isn't) — the loudest signal that your intent didn't settle them.")
        if evidence_notes:
            print("\nEvidence for split rows:")
            print("\n".join(evidence_notes))
        print(f"\n  LLM-judged plans call `claude -p ... --model {JUDGE_MODEL}` against the "
              "full plan text and are cached by file content hash in "
              "artifacts/m1/.judge_cache.json, so re-running this cell is free unless a plan "
              "changed. A judge call can still misjudge; read the evidence quotes for anything "
              "surprising rather than trusting the table blindly.")


### See the variance

The cell above prints the numbers; the cell below *draws* them. It renders a **decision heatmap** — one row per candidate gap, one column per model — so you can read your intent's coverage at a glance. Both cells grade gaps the same way: by asking Claude Code to read each full plan and judge it, not by scanning for keywords.

- A row that's **one solid color across every model** is a decision your   intent pinned tightly enough that the model choice stopped mattering.
- A **multi-color row** is a gap where the models diverged — an assumption   you left open.
- A **grey `—`** is worse than divergence: that model never wrote the   decision down at all, so it's running on a default nobody can see.

Reading left to right, a weaker model's column tends to carry more grey — that's the capability difference showing up as *silence on gaps a stronger model surfaced*. Below the heatmap, the **two-axis panel** splits pure sampling noise (one model, re-run) from systematic model divergence, so you can tell luck apart from an unpinned assumption.


In [ ]:
# Visual variance map. Renders in Jupyter/nbconvert (uses IPython's HTML only —
# no extra packages beyond the `claude` CLI calls the judge makes when available).
# Self-contained: it re-reads the plan files and the judge cache so it works
# even if you skipped the text self-check above. Judging prefers Claude reading
# the full plan text over a keyword scan, falling back to keywords only if the
# `claude` CLI is unavailable/unauthenticated — the evidence quotes in the
# previous cell are the ground truth; treat a surprising cell as "go read the
# plan," not as gospel.
from IPython.display import HTML, display
import pathlib, html as _html, subprocess, json, hashlib, os

m1 = pathlib.Path(proj) / "artifacts" / "m1"

def _load_dotenv(path):
    """Load KEY=VALUE lines from a .env file into os.environ (without
    overwriting anything already set), so subprocess calls to `claude`
    inherit ANTHROPIC_API_KEY even though Jupyter itself didn't load it."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

_load_dotenv(pathlib.Path(proj) / ".env")

PREFERRED = ["baseline", "opus", "sonnet", "haiku"]
def _label(p):
    if p.name == "plan.md":  return "run1"
    if p.name == "plan.2.md": return "run2"
    return p.name[5:-3] or "plan"
def _key(p):
    lab = _label(p); base = lab.split(".")[0]
    r = PREFERRED.index(base) if base in PREFERRED else len(PREFERRED)
    return (r, lab)

_mp = sorted((p for p in m1.glob("plan.*.md") if p.name != "plan.2.md"), key=_key)
if _mp:
    plans = _mp + ([m1 / "plan.2.md"] if (m1 / "plan.2.md").exists() else [])
else:
    plans = [p for p in (m1 / "plan.md", m1 / "plan.2.md") if p.exists()]

if len(plans) < 2:
    display(HTML("<p><b>Need at least two plans to visualize.</b> Produce "
                 "<code>plan.opus.md</code>, <code>plan.sonnet.md</code>, "
                 "<code>plan.haiku.md</code> (plus optionally <code>plan.baseline.md</code>, "
                 "or legacy <code>plan.md</code> + <code>plan.2.md</code>) in "
                 "<code>artifacts/m1/</code>.</p>"))
else:
    labels = [_label(p) for p in plans]

    JUDGE_MODEL = "opus"
    CACHE_PATH = m1 / ".judge_cache.json"
    GAP_SPECS = [
        ("unit", "What unit is TRANSFER_LIMIT measured in?",
         [("dollars", ["$", "dollar", "cent"]),
          ("count", ["count", "number of", "per transfer", "transfers per"])]),
        ("currency", "Does the limit apply to a single currency or multiple?",
         [("USD-only", ["usd", "us dollar", "dollar only", "single currency"]),
          ("multi-currency", ["multi-currenc", "multiple currenc", "eur", "settlement currenc"])]),
        ("cross-border", "Does an international transfer count differently than a domestic one?",
         [("domestic-only", ["domestic only", "no cross-border", "not international"]),
          ("counts-the-same", ["cross-border", "cross border", "international"]),
          ("differs-by-country", ["by country", "by-country", "region"])]),
        ("holds/posted", "Do authorization holds count toward the limit, or only posted transfers?",
         [("on-hold", ["when placed", "at authorization", "on placement", "hold"]),
          ("on-posted", ["posted", "posting", "once posted", "ledger"])]),
        ("scope", "Is the limit per-account, or per-partner (covering all of a partner's accounts)?",
         [("per-partner", ["per-partner", "per partner", "partner api key", "partner key", "api key"]),
          ("per-account", ["per-account", "per account", "single account"])]),
        ("window", "Is the limit a rolling window, or does it reset on a calendar boundary?",
         [("rolling", ["rolling", "24-hour", "24 hour", "last 24"]),
          ("calendar", ["calendar", "midnight", "resets"])]),
        ("weekend", "Does the plan address weekend/settlement-day timing edge cases?",
         [("addressed", ["weekend", "settlement day", "business day", "friday", "monday"])]),
    ]

    def _load_cache():
        if CACHE_PATH.exists():
            try:
                return json.loads(CACHE_PATH.read_text())
            except Exception:
                return {}
        return {}

    def _save_cache(cache):
        CACHE_PATH.write_text(json.dumps(cache, indent=2))

    def _run_claude_judge(plan_text, model):
        gap_lines = "\n".join(
            f'{i+1}. "{name}" — {question} Valid answers: '
            f'{", ".join(opt for opt, _ in options)}, or other.'
            for i, (name, question, options) in enumerate(GAP_SPECS)
        )
        prompt = (
            "You are grading a technical plan for a payments feature (transfer limits) on "
            "whether it explicitly makes each of the following decisions. Judge ONLY from "
            "what the plan text actually says — do not assume or infer beyond it.\n\n"
            f"{gap_lines}\n\n"
            "Respond with ONLY a JSON object (no markdown fences, no other text). For each "
            "gap name above, include a key with this exact shape:\n"
            '{"addressed": true|false, "decision": "<one of the valid answers, or a short '
            "phrase if none fit>\", \"evidence\": \"<short quote or 1-sentence paraphrase from "
            "the plan, or '' if not addressed>\"}\n\n"
            "Plan text:\n<<<\n" + plan_text + "\n>>>"
        )
        proc = subprocess.run(
            ["claude", "-p", prompt, "--model", model, "--output-format", "json"],
            capture_output=True, text=True, timeout=120,
        )
        if proc.returncode != 0:
            raise RuntimeError(f"claude exited {proc.returncode}: {proc.stderr.strip()[:300]}")
        try:
            envelope = json.loads(proc.stdout)
            raw = envelope["result"] if isinstance(envelope, dict) and "result" in envelope else envelope
        except json.JSONDecodeError:
            raw = proc.stdout
        text = raw if isinstance(raw, str) else json.dumps(raw)
        text = text.strip()
        if text.startswith("```"):
            text = text.split("\n", 1)[1] if "\n" in text else text
            if text.endswith("```"):
                text = text[:-3]
            text = text.strip()
        return json.loads(text)

    def _keyword_judge(plan_text):
        """Deterministic fallback: first keyword match per gap, else silent.
        Lower confidence than the LLM judge — paraphrases the keyword list
        doesn't cover will read as silent even if the plan addressed the gap."""
        low = plan_text.lower()
        out = {}
        for name, _, options in GAP_SPECS:
            decision = ""
            for opt, kws in options:
                if any(k in low for k in kws):
                    decision = opt
                    break
            out[name] = {"addressed": bool(decision), "decision": decision, "evidence": ""}
        return out

    def judge_plan(path):
        text = path.read_text()
        key = f"{path.name}:{hashlib.sha256(text.encode()).hexdigest()[:16]}"
        cache = _load_cache()
        if key in cache:
            return cache[key]
        try:
            result = _run_claude_judge(text, JUDGE_MODEL)
            result["_method"] = "llm"
        except FileNotFoundError:
            result = _keyword_judge(text)
            result["_method"] = "keyword"
            result["_fallback_reason"] = "`claude` CLI not found on PATH"
        except Exception as e:
            result = _keyword_judge(text)
            result["_method"] = "keyword"
            result["_fallback_reason"] = str(e)[:200]
        cache[key] = result
        _save_cache(cache)
        return result

    results, errors = {}, {}
    for lab, p in zip(labels, plans):
        try:
            results[lab] = judge_plan(p)
        except Exception as e:
            errors[lab] = str(e)

    usable = [l for l in labels if l in results]
    if errors:
        display(HTML(
            "<div style='background:#fff3cd;border:1px solid #ffe69c;padding:8px 12px;"
            "border-radius:4px;font:13px sans-serif;max-width:640px'>"
            "<b>&#9888; Could not judge:</b> "
            + ", ".join(f"<code>{_html.escape(l)}</code> ({_html.escape(e)})" for l, e in errors.items())
            + "</div>"
        ))
    fallback_labels = [l for l in usable if results[l].get("_method") == "keyword"]
    if fallback_labels:
        display(HTML(
            "<div style='background:#e7f1ff;border:1px solid #b6d4fe;padding:8px 12px;"
            "border-radius:4px;font:13px sans-serif;max-width:640px'>"
            "<b>Note:</b> " + ", ".join(f"<code>{_html.escape(l)}</code>" for l in fallback_labels)
            + " judged by keyword scan, not the LLM judge (claude CLI unavailable/"
            "unauthenticated). Keyword results are lower-confidence."
            + "</div>"
        ))
    if len(usable) < 2:
        display(HTML("<p><b>Fewer than two plans could be judged — skipping the heatmap.</b></p>"))
    else:
        SILENT = "—"  # em dash = the plan never addressed this decision

        def decide(lab, name):
            g = results[lab].get(name, {})
            return g.get("decision") if g.get("addressed") else SILENT

        grid = {name: [decide(l, name) for l in usable] for name, _, _ in GAP_SPECS}

        PAL = ["#cfe8ff", "#d6f5d6", "#ffe0b2", "#f3d1ff", "#ffd6d6", "#d9f2f0"]
        def row_colors(vals):
            seen = {}; i = 0; out = []
            for v in vals:
                if v == SILENT:
                    out.append(("#e6e6e6", "#8a8a8a")); continue
                if v not in seen:
                    seen[v] = PAL[i % len(PAL)]; i += 1
                out.append((seen[v], "#111"))
            return out

        def variance(vals):
            distinct = len(set(vals))
            silent = vals.count(SILENT)
            return distinct, silent

        order = sorted(GAP_SPECS, key=lambda g: variance(grid[g[0]]), reverse=True)

        cell = "padding:5px 9px;border:1px solid #bbb;text-align:center;font:13px monospace;"
        head = "padding:5px 9px;border:1px solid #bbb;font:600 12px sans-serif;background:#333;color:#fff;"
        rows = ["<tr><th style='" + head + "text-align:left'>gap</th>"]
        rows += ["<th style='" + head + "'>" + _html.escape(l)
                 + (" *" if l in fallback_labels else "") + "</th>" for l in usable]
        rows.append("<th style='" + head + "'>variance</th></tr>")
        body = []
        for name, _, _ in order:
            vals = grid[name]; cols = row_colors(vals)
            tds = ["<td style='" + cell + "text-align:left;font-weight:600;background:#333;color:#fff'>"
                   + _html.escape(name) + "</td>"]
            for v, (bg, fg) in zip(vals, cols):
                tds.append("<td style='" + cell + "background:" + bg + ";color:" + fg + "'>"
                           + _html.escape(v) + "</td>")
            distinct, silent = variance(vals)
            swatches = "".join("<span style='display:inline-block;width:12px;height:12px;margin:1px;"
                               "border:1px solid #999;background:" + bg + "'></span>"
                               for bg, _ in dict.fromkeys(cols))
            note = " " + str(distinct) + " answer" + ("s" if distinct != 1 else "")
            note += (" (" + str(silent) + " silent)") if silent else ""
            tds.append("<td style='" + cell + "text-align:left'>" + swatches + note + "</td>")
            body.append("<tr>" + "".join(tds) + "</tr>")

        caption = ("<p style='font:13px sans-serif;margin:0 0 6px'>"
                   "<b>Decision heatmap.</b> Colors are matched <i>within each row</i>: "
                   "same color = models agreed on that decision, different colors = they diverged, "
                   "grey <code>&mdash;</code> = the model never wrote it down (a hidden default). "
                   f"Judged by Claude ({JUDGE_MODEL}) reading the full plan text where available "
                   "(cached by content hash — re-running costs nothing unless a plan changed); "
                   "columns marked <code>*</code> fell back to a keyword scan. Rows are sorted "
                   "loudest-first."
                   + (" The <code>baseline</code> column has no intent doc behind it — expect it to "
                      "diverge; that contrast is the point." if "baseline" in usable else "")
                   + "</p>")
        display(HTML(caption + "<table style='border-collapse:collapse'>"
                     + "".join(rows) + "".join(body) + "</table>"))

        # --- axis panels: sampling noise, systematic model divergence, and the ---
        # --- intent doc's own effect (with vs. without one) ----------------------
        def base_of(lab): return lab.split(".")[0]

        # Reference plan for all axis comparisons: prefer an intent-driven model
        # plan, never the no-intent baseline itself.
        base_lab = next((l for l in ("opus", "sonnet", "haiku") if l in usable), None)
        if base_lab is None:
            base_lab = next((l for l in usable if l != "baseline"), usable[0])
        base_model = base_of(base_lab)
        sampling_partner = next((l for l in usable if base_of(l) == base_model and l != base_lab), None)
        model_partner = None
        for pref in ("haiku", "sonnet"):
            model_partner = next((l for l in usable if base_of(l) == pref and pref != base_model), None)
            if model_partner: break
        if not model_partner:
            model_partner = next((l for l in usable if base_of(l) not in (base_model, "baseline")), None)
        baseline_partner = "baseline" if "baseline" in usable and "baseline" != base_lab else None

        def axis_rows(partner):
            out = []; agree_n = 0
            for name, _, _ in GAP_SPECS:
                va = grid[name][usable.index(base_lab)]; vb = grid[name][usable.index(partner)]
                ok = va == vb; agree_n += ok
                mark = "&#10003;" if ok else "&#10007;"
                col = "#2e7d32" if ok else "#c62828"
                out.append((name, mark, col, va, vb))
            return out, agree_n

        panels = []
        if sampling_partner:
            panels.append(("Sampling axis &mdash; " + base_lab + " vs " + sampling_partner
                           + " (same model, re-run)", sampling_partner,
                           "Disagreements here are <b>noise</b>: the model rolled the dice differently."))
        if model_partner:
            panels.append(("Model axis &mdash; " + base_lab + " vs " + model_partner
                           + " (swap the model)", model_partner,
                           "Disagreements here are <b>systematic</b>: each model's own default leaking in."))
        if baseline_partner:
            panels.append(("Intent axis &mdash; " + base_lab + " vs baseline "
                           + "(with vs. without an intent doc)", baseline_partner,
                           "Disagreements here show what the intent doc actually bought you: if this "
                           "panel agrees <b>less</b> than the model axis above, the intent doc is doing "
                           "real work — not just theater."))

        if panels:
            blocks = []
            for title, partner, gloss in panels:
                arows, agree_n = axis_rows(partner)
                trs = []
                for name, mark, col, va, vb in arows:
                    trs.append("<tr><td style='" + cell + "text-align:left'>" + _html.escape(name)
                               + "</td><td style='" + cell + "color:" + col + ";font-weight:700'>"
                               + mark + "</td><td style='" + cell + "font-size:11px'>"
                               + _html.escape(va) + " / " + _html.escape(vb) + "</td></tr>")
                hdr = ("<tr><th style='" + head + "text-align:left'>gap</th><th style='" + head
                       + "'>=?</th><th style='" + head + "'>" + _html.escape(base_lab) + " / "
                       + _html.escape(partner) + "</th></tr>")
                blocks.append("<div style='display:inline-block;vertical-align:top;margin:0 18px 8px 0'>"
                              "<p style='font:600 13px sans-serif;margin:0'>" + title + "</p>"
                              "<p style='font:12px sans-serif;margin:2px 0 6px;color:#555'>" + gloss
                              + " Agree on <b>" + str(agree_n) + "/" + str(len(GAP_SPECS))
                              + "</b> gaps.</p><table style='border-collapse:collapse'>"
                              + hdr + "".join(trs) + "</table></div>")
            note = ""
            if not sampling_partner:
                note = ("<p style='font:12px sans-serif;color:#555'>Tip: add "
                        "<code>plan." + base_model + ".2.md</code> (a second run of the same model) "
                        "to see the sampling axis beside the others.</p>")
            display(HTML("<h4 style='font:700 14px sans-serif;margin:14px 0 4px'>Axes of variance"
                         "</h4>" + note + "<div>" + "".join(blocks) + "</div>"
                         "<p style='font:12px sans-serif;color:#555;max-width:640px'>The sampling panel "
                         "should mostly agree; the model panel usually agrees less; the intent panel "
                         "(baseline vs. an intent-driven plan) should agree the least of all — that's the "
                         "measured value of writing the intent doc in the first place.</p>"))


## 3 · Debrief

- Which rows in the gap matrix were marked `<- split`? For each, decide: is   that a decision your intent *should* have pinned, or one it's fine to   defer to implementation?
- Which disagreements showed up between Haiku, Sonnet, and Opus (the model   axis) versus between two runs of the same model (the sampling axis)? A   model-axis split is the stronger signal — trace it back to the exact line   your intent left open.
- Did the stronger model quietly pin gaps the weaker one skipped (currency,   window, per-partner scope)? That's a capability difference — but it's also   a gap *your* intent should have closed so the model didn't have to.
- Is an un-pinned unit (like `TRANSFER_LIMIT`'s dollars-vs-count-vs-window   ambiguity) a bug in your intent doc, or sometimes a legitimate preference   you can defer? How would you tell the difference?
- If all three plans agreed on everything, does that mean your intent pinned   every gap — or did the models happen to share the same default? How would   you check which one happened?
- Look at `plan.baseline.md` against your intent-driven plans (the "intent   axis" panel). How many more gaps did the baseline leave silent or resolve   differently, compared to how much the opus/sonnet/haiku plans agree with   *each other*? That difference is the actual, measured value of the intent   doc — not a hypothetical.


---
### Take-home

Intent pins decisions; variance measures what you left to chance. The more precisely your intent doc pins the ambiguous points, the less independent planning runs disagree — whether you re-run one model or swap Haiku for Opus. Any disagreement that remains is a map of exactly what you left unpinned, and a disagreement that tracks the *model* rather than the *run* is the loudest entry on that map. A better model can hide an unpinned gap; only the intent doc closes it.

(Related concept: a reusable command captures this process so you don't have to re-explain it each time — see Module 2.)
